# Conditioning and Uncertainty-Aware Routing

This notebook evaluates the Phase 0 architecture on KF12.

The experiments proceed in three stages:

1. **Phase 0A:** replace the MLP history encoder with a Neural CDE.
2. **Phase 0A robustness:** confirm the CDE improvement across paired random seeds.
3. **Phase 0B:** test whether evidential regime uncertainty provides additional value through mixture-of-experts routing.

The final Phase 0 model retains only components supported by validation results. The test split remains untouched.

## Setup

In [1]:
import os
import subprocess

REPO_PATH = "/kaggle/working/diffusion-portfolio"

REPO_URL = (
    "https://github.com/"
    "Amir-Entezari/"
    "diffusion-portfolio.git"
)

if not os.path.exists(
    os.path.join(
        REPO_PATH,
        ".git",
    )
):
    subprocess.run(
        [
            "git",
            "clone",
            REPO_URL,
            REPO_PATH,
        ],
        check=True,
    )

subprocess.run(
    [
        "git",
        "-C",
        REPO_PATH,
        "fetch",
        "origin",
        "main",
    ],
    check=True,
)

subprocess.run(
    [
        "git",
        "-C",
        REPO_PATH,
        "reset",
        "--hard",
        "origin/main",
    ],
    check=True,
)

CURRENT_COMMIT = subprocess.check_output(
    [
        "git",
        "-C",
        REPO_PATH,
        "rev-parse",
        "HEAD",
    ],
    text=True,
).strip()

os.chdir(
    REPO_PATH
)

print(
    "Synced to latest main:"
)
print(
    CURRENT_COMMIT
)

subprocess.run(
    [
        "git",
        "log",
        "-1",
        "--oneline",
    ],
    check=True,
)

Cloning into '/kaggle/working/diffusion-portfolio'...


HEAD is now at cf86f15 Make Phase 0 training safely resumable
Synced to latest main:
cf86f158948520099a343bdbd3ec8ee0ddcb7b34
cf86f15 Make Phase 0 training safely resumable


From https://github.com/Amir-Entezari/diffusion-portfolio
 * branch            main       -> FETCH_HEAD


CompletedProcess(args=['git', 'log', '-1', '--oneline'], returncode=0)

In [2]:
!pip install -q -e '.[cde]'

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 2.6 MB/s eta 0:00:00
  Building editable for diffusion-portfolio (pyproject.toml) ... done


### Tests

In [3]:
!pytest -q \
    tests/unit/test_cde.py \
    tests/unit/test_conditional_diffusion_model.py \
    tests/unit/test_diffusion_training.py \
    tests/unit/test_evidential.py \
    tests/unit/test_evidential_regimes.py

........................................                                 [100%]


### Restore

Restore any available previous experiment outputs. Missing backups are skipped; the corresponding experiments will run from scratch.

In [6]:
from pathlib import Path
import shutil
import kagglehub

HANDLE = "amirentezari/diffusion-portfolio"

RESTORES = {
    "baselines/vanilla_vpred":
        "/kaggle/working/vanilla_seed42",

    "phase0/neural_cde_rk4x4":
        "/kaggle/working/phase0_cde",

    "phase0/multiseed":
        "/kaggle/working/phase0_multiseed",

    "phase0/evidential_next_day":
        "/kaggle/working/phase0_evidential",

    "phase0/evidential_current":
        "/kaggle/working/phase0_evidential_current",

    "phase0/router":
        "/kaggle/working/phase0_router",
}

try:
    artifact_path = Path(
        kagglehub.dataset_download(
            HANDLE
        )
    )

    for relative, destination in (
        RESTORES.items()
    ):
        source = (
            artifact_path
            / relative
        )

        destination = Path(
            destination
        )

        if destination.exists():
            print(
                "Already available:",
                destination,
            )

        elif source.exists():
            shutil.copytree(
                source,
                destination,
            )

            print(
                "Restored:",
                destination,
            )

        else:
            print(
                "No backup:",
                relative,
            )

except Exception as error:
    print(
        "No backups restored."
    )
    print(
        "Experiments will run from scratch."
    )
    print(error)

Restored: /kaggle/working/vanilla_seed42
Restored: /kaggle/working/phase0_cde
Restored: /kaggle/working/phase0_multiseed
Already available: /kaggle/working/phase0_evidential
Restored: /kaggle/working/phase0_evidential_current
Restored: /kaggle/working/phase0_router


### Vanilla seed-42 reference

In [ ]:
!PYTHONUNBUFFERED=1 python -u scripts/train_diffusion.py \
    --config configs/mvp.yaml \
    --output-dir /kaggle/working/vanilla_seed42 \
    --device cuda \
    --resume

In [ ]:
!PYTHONUNBUFFERED=1 python -u scripts/evaluate_diffusion.py \
    --run-dir /kaggle/working/vanilla_seed42 \
    --batch-size 32 \
    --device cuda

## Neural CDE Conditioning

Phase 0A tests whether replacing the Vanilla diffusion model's MLP history encoder with a Neural CDE improves probabilistic forecasting on KF12.

### Canonical seed-42 training

In [9]:
!PYTHONUNBUFFERED=1 python -u scripts/train_diffusion.py \
    --config configs/cde.yaml \
    --output-dir /kaggle/working/phase0_cde \
    --device cuda \
    --resume

VANILLA CONDITIONAL DIFFUSION TRAINING
Device: cuda
GPU: Tesla T4
Prediction type: v_prediction
History encoder: cde
Diffusion steps: 100

Loading KF12 data...
Train samples: 10516
Validation samples: 1256
Test samples: 4781
Parameters: 1,321,793

Training...
Epoch 001/050 | train=0.979311 | val=1.819148
Epoch 002/050 | train=0.844835 | val=1.786172
Epoch 003/050 | train=0.796894 | val=1.700542
Epoch 004/050 | train=0.782263 | val=1.659349
Epoch 005/050 | train=0.759299 | val=1.591604
Epoch 006/050 | train=0.739555 | val=1.595205
Epoch 007/050 | train=0.745646 | val=1.578609
Epoch 008/050 | train=0.721117 | val=1.591480
Epoch 009/050 | train=0.751829 | val=1.558710
Epoch 010/050 | train=0.747048 | val=1.559329
Epoch 011/050 | train=0.715129 | val=1.572181
Epoch 012/050 | train=0.712936 | val=1.546897
Epoch 013/050 | train=0.734503 | val=1.545789
Epoch 014/050 | train=0.710157 | val=1.541380
Epoch 015/050 | train=0.735714 | val=1.562583
Epoch 016/050 | train=0.726755 | val=1.549307
Epoc

### Validation evaluation

In [6]:
!PYTHONUNBUFFERED=1 python -u scripts/evaluate_diffusion.py \
    --run-dir /kaggle/working/phase0_cde \
    --batch-size 32 \
    --device cuda

DIFFUSION VALIDATION FORECAST EVALUATION
Device: cuda
GPU: Tesla T4
History encoder: cde
Scenarios per date: 256
Inference batch size: 32
Sampling seed: 12345

Validation samples: 1256
Validation start: 2000-01-03 00:00:00
Validation end: 2004-12-31 00:00:00

Checkpoint epoch: 42
Checkpoint validation loss: 1.445402071734143

Generating VALIDATION scenarios...
Validation sampling: 100%|██████████████████████| 40/40 [02:37<00:00,  3.93s/it]

Saved validation scenarios: /kaggle/working/phase0_cde/validation_scenarios.npz
Scenario cube: (1256, 256, 12)
Raw scenario mean: -0.0010567344725131989
Raw scenario std: 0.013828642666339874
Raw scenario max abs: 0.1086999848484993

Computing VALIDATION probabilistic metrics...

VALIDATION PROBABILISTIC EVALUATION
CRPS mean: 0.008184307740418832
CRPS std across assets: 0.0018269477406119233
Energy Score: 0.03376962808211297

Calibration:
   level         PICP          ACE
    0.50     0.429472    -0.070528
    0.80     0.720276    -0.079724
    0.9

### Canonical seed-42 verification

In [17]:
import json
import pandas as pd
import torch

RUN_DIR = "/kaggle/working/phase0_cde"

summary = json.load(open(f"{RUN_DIR}/summary.json"))
history = pd.read_csv(f"{RUN_DIR}/history.csv")
checkpoint = torch.load(
    f"{RUN_DIR}/best.pt",
    map_location="cpu",
    weights_only=False,
)

print("epochs:", len(history))
print("summary best:", summary["best_epoch"], summary["best_val_loss"])
print("checkpoint:", checkpoint["epoch"], checkpoint["val_loss"])

assert len(history) == 50
assert checkpoint["epoch"] == summary["best_epoch"]
assert abs(
    checkpoint["val_loss"]
    - summary["best_val_loss"]
) < 1e-8

print("✅ checkpoint is consistent")

epochs: 50
summary best: 42 1.445402071734143
checkpoint: 42 1.445402071734143
✅ checkpoint is consistent


### Vanilla vs Neural CDE — Seed 42

In [19]:
import json
from pathlib import Path

import pandas as pd

VANILLA_42 = Path(
    "/kaggle/working/vanilla_seed42"
)

CDE_42 = Path(
    "/kaggle/working/phase0_cde"
)

with (
    VANILLA_42
    / "validation_probabilistic_metrics.json"
).open(
    encoding="utf-8"
) as f:
    vanilla = json.load(f)

with (
    VANILLA_42
    / "summary.json"
).open(
    encoding="utf-8"
) as f:
    vanilla_summary = json.load(f)

with (
    CDE_42
    / "validation_probabilistic_metrics.json"
).open(
    encoding="utf-8"
) as f:
    cde = json.load(f)

with (
    CDE_42
    / "summary.json"
).open(
    encoding="utf-8"
) as f:
    cde_summary = json.load(f)

comparison = pd.DataFrame({
    "Vanilla MLP": [
        vanilla_summary["best_val_loss"],
        vanilla["crps_mean"],
        vanilla["energy_score"],
    ],
    "Neural CDE": [
        cde_summary["best_val_loss"],
        cde["crps_mean"],
        cde["energy_score"],
    ],
}, index=[
    "Validation diffusion loss ↓",
    "CRPS ↓",
    "Energy Score ↓",
])

comparison["Change %"] = (
    (
        comparison["Neural CDE"]
        - comparison["Vanilla MLP"]
    )
    / comparison["Vanilla MLP"]
    * 100
)

comparison

,Vanilla MLP,Neural CDE,Change %
Validation diffusion loss ↓,1.579357,1.445402,-8.481630
CRPS ↓,0.008356,0.008184,-2.051135
Energy Score ↓,0.034616,0.033770,-2.443752


In [20]:
vanilla_cal = pd.DataFrame(
    vanilla["calibration"]
)

cde_cal = pd.DataFrame(
    cde["calibration"]
)

calibration_comparison = pd.DataFrame({
    "Level": vanilla_cal["level"],
    "Vanilla PICP": vanilla_cal["picp"],
    "CDE PICP": cde_cal["picp"],
    "Vanilla |ACE|": vanilla_cal["ace"].abs(),
    "CDE |ACE|": cde_cal["ace"].abs(),
})

calibration_comparison

,Level,Vanilla PICP,CDE PICP,Vanilla |ACE|,CDE |ACE|
0,0.50,0.323381,0.429472,0.176619,0.070528
1,0.80,0.578357,0.720276,0.221643,0.079724
2,0.90,0.700637,0.831940,0.199363,0.068060
3,0.95,0.784833,0.895701,0.165167,0.054299
4,0.99,0.880507,0.953954,0.109493,0.036046


### Multi-Seed Robustness

The seed-42 improvement is confirmed using paired Vanilla and Neural CDE runs for seeds 42, 43, and 44 under the same validation protocol.

In [18]:
!PYTHONUNBUFFERED=1 python -u scripts/run_multiseed.py \
    --seed42-vanilla-dir /kaggle/working/vanilla_seed42 \
    --seed42-cde-dir /kaggle/working/phase0_cde \
    --output-root /kaggle/working/phase0_multiseed \
    --seeds 43 44 \
    --device cuda

PHASE 0A — PAIRED MULTI-SEED CONFIRMATION
Existing seed 42 Vanilla: /kaggle/input/datasets/amirentezari/diffusion-portfolio/baselines/vanilla_vpred
Existing seed 42 CDE: /kaggle/working/phase0_cde
New paired seeds: [43, 44]
Training already complete: vanilla seed 43
Validation evaluation already complete: vanilla seed 43

TRAINING CDE SEED 43

$ /usr/bin/python3 scripts/train_diffusion.py --config /kaggle/working/phase0_multiseed/configs/cde_seed_43.yaml --output-dir /kaggle/working/phase0_multiseed/cde/seed_43 --device cuda
VANILLA CONDITIONAL DIFFUSION TRAINING
Device: cuda
GPU: Tesla T4
Prediction type: v_prediction
History encoder: cde
Diffusion steps: 100

Loading KF12 data...
Train samples: 10516
Validation samples: 1256
Test samples: 4781
Parameters: 1,321,793

Training...
Epoch 001/050 | train=0.969969 | val=1.805655
Epoch 002/050 | train=0.858277 | val=1.740492
Epoch 003/050 | train=0.794634 | val=1.723418
Epoch 004/050 | train=0.780842 | val=1.647758
Epoch 005/050 | train=0.8

In [ ]:
import json
from pathlib import Path

import pandas as pd

MULTISEED_DIR = Path(
    "/kaggle/working/phase0_multiseed"
)

per_seed = pd.read_csv(
    MULTISEED_DIR
    / "per_seed_results.csv"
)

with (
    MULTISEED_DIR
    / "summary.json"
).open(
    encoding="utf-8"
) as f:
    multiseed_summary = json.load(f)

display(
    per_seed
)

paired = pd.DataFrame(
    multiseed_summary[
        "summary"
    ][
        "paired"
    ]
).T

display(
    paired
)

### conclusion

Neural CDE outperforms Vanilla on validation diffusion loss, CRPS, Energy Score, and mean absolute calibration error for all three paired seeds.

The Neural CDE is therefore retained as the Phase 0 conditioning backbone.

## Evidential Regime Uncertainty

Phase 0B tests whether uncertainty derived from the Neural CDE representation provides useful regime information and whether that signal improves downstream diffusion forecasting through uncertainty-aware routing.

### Phase 0B1a — Next-Day Regime Probe

In [24]:
!python scripts/train_evidential_regime.py \
    --cde-run-dir /kaggle/working/phase0_cde \
    --config configs/evidential.yaml \
    --output-dir /kaggle/working/phase0_evidential \
    --regime-target next_day \
    --device cuda

PHASE 0B1 — FROZEN CDE REGIME PROBES
Device: cuda
Frozen CDE checkpoint epoch: 42
Frozen CDE validation loss: 1.445402071734143

Extracting frozen CDE embeddings...
Train embeddings: (10516, 128)
Validation embeddings: (1256, 128)

Train-only thresholds: (0.004863632811078985, 0.008021274431265364)
Train class counts: [3505, 3506, 3505]
Validation class counts: [98, 279, 879]

Training evidential probe...
Evidential 001/050 | train=1.596359 | val_nll=1.168709 | kl=0.100
Evidential 002/050 | train=1.611986 | val_nll=1.208462 | kl=0.200
Evidential 003/050 | train=1.599471 | val_nll=1.222922 | kl=0.300
Evidential 004/050 | train=1.579625 | val_nll=1.219139 | kl=0.400
Evidential 005/050 | train=1.608150 | val_nll=1.208724 | kl=0.500
Evidential 006/050 | train=1.595672 | val_nll=1.198392 | kl=0.600
Evidential 007/050 | train=1.595911 | val_nll=1.183058 | kl=0.700
Evidential 008/050 | train=1.596618 | val_nll=1.174049 | kl=0.800
Evidential 009/050 | train=1.601604 | val_nll=1.164515 | kl=0.9

In [26]:
import numpy as np
import pandas as pd

path = (
    "/kaggle/working/phase0_evidential/"
    "validation_predictions.npz"
)

data = np.load(path)

labels = data["labels"]
evidential_probs = data[
    "evidential_probabilities"
]
softmax_probs = data[
    "softmax_probabilities"
]
vacuity = data[
    "evidential_vacuity"
]

evidential_pred = (
    evidential_probs.argmax(axis=1)
)

softmax_pred = (
    softmax_probs.argmax(axis=1)
)

def confusion(
    truth,
    prediction,
):
    return pd.crosstab(
        pd.Series(
            truth,
            name="True",
        ),
        pd.Series(
            prediction,
            name="Predicted",
        ),
        dropna=False,
    ).reindex(
        index=[0, 1, 2],
        columns=[0, 1, 2],
        fill_value=0,
    )

print("Evidential confusion matrix")
display(
    confusion(
        labels,
        evidential_pred,
    )
)

print("Softmax confusion matrix")
display(
    confusion(
        labels,
        softmax_pred,
    )
)

rows = []

for regime in [0, 1, 2]:
    mask = (
        labels == regime
    )

    rows.append({
        "regime": regime,
        "count": int(
            mask.sum()
        ),
        "evidential_recall": float(
            (
                evidential_pred[mask]
                == regime
            ).mean()
        ),
        "softmax_recall": float(
            (
                softmax_pred[mask]
                == regime
            ).mean()
        ),
        "mean_vacuity": float(
            vacuity[mask].mean()
        ),
    })

display(
    pd.DataFrame(rows)
)

print(
    "Always-stressed accuracy:",
    float(
        (labels == 2).mean()
    ),
)

print(
    "Evidential prediction counts:",
    np.bincount(
        evidential_pred,
        minlength=3,
    ),
)

print(
    "Softmax prediction counts:",
    np.bincount(
        softmax_pred,
        minlength=3,
    ),
)

Evidential confusion matrix


Predicted,0,1,2
True,,,
0,35,29,34
1,84,90,105
2,248,268,363


Softmax confusion matrix


Predicted,0,1,2
True,,,
0,23,41,34
1,85,110,84
2,274,299,306


,regime,count,evidential_recall,softmax_recall,mean_vacuity
0,0,98,0.357143,0.234694,0.752505
1,1,279,0.322581,0.394265,0.745102
2,2,879,0.412969,0.348123,0.707001


Always-stressed accuracy: 0.6998407643312102
Evidential prediction counts: [367 387 502]
Softmax prediction counts: [382 450 424]


### Phase 0B1b — Current-Regime Probe

Because next-day regime classification is strongly affected by the validation-period regime shift, the second probe targets the current regime using recent realized market behavior.

In [29]:
!python scripts/train_evidential_regime.py \
    --cde-run-dir /kaggle/working/phase0_cde \
    --config configs/evidential.yaml \
    --output-dir /kaggle/working/phase0_evidential_current \
    --regime-target current \
    --device cuda

PHASE 0B1 — FROZEN CDE REGIME PROBES
Device: cuda
Frozen CDE checkpoint epoch: 42
Frozen CDE validation loss: 1.445402071734143

Extracting frozen CDE embeddings...
Train embeddings: (10516, 128)
Validation embeddings: (1256, 128)

Train-only thresholds: (0.006870479894586817, 0.008949539408943621)
Train class counts: [3505, 3506, 3505]
Validation class counts: [9, 220, 1027]

Training evidential probe...
Evidential 001/050 | train=1.584944 | val_nll=1.181105 | kl=0.100
Evidential 002/050 | train=1.596790 | val_nll=1.284037 | kl=0.200
Evidential 003/050 | train=1.578518 | val_nll=1.235005 | kl=0.300
Evidential 004/050 | train=1.573105 | val_nll=1.257824 | kl=0.400
Evidential 005/050 | train=1.572171 | val_nll=1.224538 | kl=0.500
Evidential 006/050 | train=1.576315 | val_nll=1.215821 | kl=0.600
Evidential 007/050 | train=1.584968 | val_nll=1.208789 | kl=0.700
Evidential 008/050 | train=1.580888 | val_nll=1.186492 | kl=0.800
Evidential 009/050 | train=1.588622 | val_nll=1.189954 | kl=0.9

In [30]:
import json
import numpy as np
import pandas as pd

RUN_DIR = "/kaggle/working/phase0_evidential_current"

data = np.load(
    f"{RUN_DIR}/validation_predictions.npz"
)

with open(
    f"{RUN_DIR}/regime_thresholds.json"
) as f:
    thresholds_info = json.load(f)

scores = data["stress_scores"]
vacuity = data["evidential_vacuity"]
labels = data["labels"]

low = thresholds_info[
    "thresholds"
]["calm_normal"]

high = thresholds_info[
    "thresholds"
]["normal_stressed"]

distance_to_boundary = np.minimum(
    np.abs(scores - low),
    np.abs(scores - high),
)

correlation = np.corrcoef(
    distance_to_boundary,
    vacuity,
)[0, 1]

print(
    "Correlation(distance to boundary, vacuity):",
    correlation,
)

frame = pd.DataFrame({
    "score": scores,
    "vacuity": vacuity,
    "distance_to_boundary": distance_to_boundary,
    "label": labels,
})

frame["boundary_distance_quartile"] = pd.qcut(
    frame["distance_to_boundary"],
    q=4,
    labels=[
        "closest",
        "near",
        "far",
        "farthest",
    ],
)

display(
    frame.groupby(
        "boundary_distance_quartile",
        observed=True,
    )[
        [
            "distance_to_boundary",
            "vacuity",
        ]
    ].mean()
)

print()
print(
    "Mean vacuity closest 25%:",
    frame.loc[
        frame["boundary_distance_quartile"]
        == "closest",
        "vacuity",
    ].mean(),
)

print(
    "Mean vacuity farthest 25%:",
    frame.loc[
        frame["boundary_distance_quartile"]
        == "farthest",
        "vacuity",
    ].mean(),
)

Correlation(distance to boundary, vacuity): -0.3375002239895621


,distance_to_boundary,vacuity
boundary_distance_quartile,,
closest,0.000569,0.770410
near,0.002658,0.693100
far,0.006006,0.655624
farthest,0.012661,0.602368



Mean vacuity closest 25%: 0.7704095
Mean vacuity farthest 25%: 0.6023676


### Phase 0B1 conclusion

Phase 0B1 tested whether the frozen Neural CDE representation from Phase 0A contains useful regime information and whether a Dirichlet evidential head produces a meaningful uncertainty signal.

The evidential head maps the CDE representation \(H_T\) to non-negative evidence

\[
e=\operatorname{Softplus}(W H_T+b), \qquad
\alpha_k=e_k+1,
\]

with Dirichlet strength and expected class probabilities

\[
S=\sum_{k=1}^{K}\alpha_k,
\qquad
p_k=\frac{\alpha_k}{S},
\]

and evidential vacuity

\[
u=\frac{K}{S}.
\]

We initially tested prediction of the **next-day stress regime** (Phase 0B1a), but the strong train/validation regime shift made this unsuitable as the routing target.

Phase 0B1b therefore used the **current market regime**, defined from the trailing 20-day cross-sectional RMS return magnitude, with calm/normal/stressed thresholds fitted only on the training period.

The hard regime classification itself was weak under the large validation-period distribution shift (validation was ~81.8% stressed). However, the evidential model showed substantially better probabilistic behavior than an equal-capacity softmax probe:

- Evidential NLL: **1.0300** vs. Softmax **1.2619**
- Evidential Brier: **0.6089** vs. Softmax **0.7175**
- Evidential ECE: **0.1268** vs. Softmax **0.1404**

More importantly, the vacuity signal behaved meaningfully:

\[
u_{\text{incorrect}}=0.7513
>
u_{\text{correct}}=0.5855.
\]

Vacuity also increased near regime-transition boundaries:

\[
\operatorname{corr}
(\text{distance to nearest boundary},u)
=-0.338,
\]

with mean vacuity **0.7704** for the closest boundary-distance quartile versus **0.6024** for the farthest quartile.

### Conclusion

Phase 0B1 does **not** establish strong hard regime classification. However, it provides evidence that the Dirichlet head extracts a meaningful uncertainty/vacuity signal from the frozen CDE representation: uncertainty rises for incorrect predictions and near regime boundaries.

Therefore Phase 0B proceeds to **Phase 0B2: uncertainty-aware routing**. The next ablation will compare:

\[
\text{CDE}
\quad\text{vs.}\quad
\text{CDE + Softmax MoE}
\quad\text{vs.}\quad
\text{CDE + Evidential MoE}.
\]

The evidential router will use

\[
\lambda=\sigma\!\left(\gamma(u-\tau)\right),
\]

\[
G_k=(1-\lambda)p_k+\lambda\frac{1}{K},
\]

so that high evidential uncertainty moves routing weights toward uniform expert weighting. The decisive evaluation will be downstream diffusion quality—validation loss, CRPS, Energy Score, and calibration—not regime-classification accuracy alone.

### Phase 0B2 — Uncertainty-Aware Routing

In [32]:
!python scripts/train_router.py \
    --cde-run-dir /kaggle/working/phase0_cde \
    --probe-run-dir /kaggle/working/phase0_evidential_current \
    --config configs/router.yaml \
    --output-dir /kaggle/working/phase0_router \
    --device cuda

PHASE 0B2 — UNCERTAINTY-AWARE ROUTING
Device: cuda
Phase 0A checkpoint: 42 1.445402071734143

Extracting frozen CDE conditions...
Train conditions: (10516, 128)                                                  
Validation conditions: (1256, 128)

Train-only tau: 0.7766247987747192

Phase 0A val loss: 1.445402071734143
Softmax initial: 1.445402071734143
Evidential initial: 1.445402071734143

Training Softmax MoE...
Softmax initial val loss: 1.445402072
Softmax 001/030 | train=0.699422 | val=1.451107
Softmax 002/030 | train=0.703936 | val=1.450023
Softmax 003/030 | train=0.683690 | val=1.450732
Softmax 004/030 | train=0.685861 | val=1.447687
Softmax 005/030 | train=0.689584 | val=1.454488
Softmax 006/030 | train=0.688629 | val=1.446289
Softmax 007/030 | train=0.693947 | val=1.443933
Softmax 008/030 | train=0.692219 | val=1.442003
Softmax 009/030 | train=0.707080 | val=1.437889
Softmax 010/030 | train=0.667213 | val=1.445402
Softmax 011/030 | train=0.689754 | val=1.442309
Softmax 012/030 

### Phase 0B2 conclusion

Phase 0B2 tested whether the evidential uncertainty signal identified in Phase 0B1 improves downstream diffusion forecasting when used for mixture-of-experts routing.

For the evidential model, routing weights were

\[
\lambda=\sigma\!\left(\gamma(u-\tau)\right),
\]

\[
G_k=(1-\lambda)p_k+\lambda\frac{1}{K},
\]

where \(p_k\) are the Dirichlet expected regime probabilities, \(u=K/S\) is evidential vacuity, and \(\tau\) was set from training-period vacuity only.

The comparison was controlled:

- the validated Phase 0A CDE and diffusion denoiser were frozen;
- the Phase 0B1 regime heads were frozen;
- Softmax MoE and Evidential MoE used identical residual experts;
- experts were zero-initialized, so both models started exactly at the Phase 0A validation loss;
- only the residual experts were trained.

### Validation results

| Metric | CDE | Softmax MoE | Evidential MoE |
|---|---:|---:|---:|
| Diffusion loss ↓ | 1.445402 | **1.437889** | 1.442055 |
| CRPS ↓ | 0.008184 | **0.008127** | 0.008134 |
| Energy Score ↓ | 0.033770 | **0.033604** | 0.033628 |
| Mean \(|ACE|\) ↓ | **0.061731** | 0.098104 | 0.097214 |

Both routed variants produced small improvements in diffusion loss, CRPS, and Energy Score, but substantially worsened calibration. More importantly, the Evidential MoE did not outperform the matched Softmax MoE on loss, CRPS, or Energy Score.

The evidential mechanism itself behaved as designed: uncertainty increased routing toward uniform expert weights, with mean uniform blending of approximately **37.7%**. Therefore the negative result is not explained by an inactive uncertainty gate.

### Decision

Phase 0B1 showed that evidential vacuity is a meaningful diagnostic signal, increasing for incorrect predictions and near regime boundaries. However, Phase 0B2 does **not** show incremental downstream value from using this uncertainty for routing.

Therefore the evidential router is **not retained in the main model**. The main architecture returns to the Phase 0A Neural CDE baseline.

\[
\boxed{\text{Retain Neural CDE; reject Evidential MoE routing}}
\]

Phase 0B remains an informative ablation rather than a component of the final architecture. Further tuning of \(\tau\), \(\gamma\), or the number of experts is deferred to avoid post-hoc hyperparameter search.

Together with the paired multi-seed confirmation of Phase 0A, this closes the Phase 0 architecture-selection stage.

## Final Phase 0 Decision

Phase 0 evaluated two extensions to the Vanilla conditional diffusion baseline.

**Neural CDE conditioning is retained.** Its improvement over Vanilla is consistent across all three paired random seeds on validation diffusion loss, CRPS, Energy Score, and calibration.

**Evidential regime routing is rejected.** Although evidential vacuity behaves as a meaningful uncertainty diagnostic, the Evidential MoE does not outperform the matched Softmax MoE and both routing variants substantially worsen calibration relative to the plain Neural CDE.

The architecture carried forward to Phase 1 is therefore:

\[
\boxed{\text{Neural CDE-conditioned diffusion}}
\]

The frozen test split remains untouched.

## Backup

Persist the completed Phase 0 experiments to the central Kaggle artifact.

In [8]:
from pathlib import Path
import shutil
import kagglehub

HANDLE = "amirentezari/diffusion-portfolio"

try:
    artifact_path = Path(
        kagglehub.dataset_download(
            HANDLE
        )
    )

except Exception as error:
    artifact_path = None

    print(
        "Existing backup unavailable:",
        error,
    )

staging = Path(
    "/kaggle/working/artifact_sync"
)

if staging.exists():
    shutil.rmtree(
        staging
    )

if artifact_path is not None:
    shutil.copytree(
        artifact_path,
        staging,
    )

else:
    staging.mkdir(
        parents=True,
        exist_ok=True,
    )

backups = {
    Path(
        "/kaggle/working/phase0_cde"
    ): (
        staging
        / "phase0"
        / "neural_cde_rk4x4"
    ),

    Path(
        "/kaggle/working/phase0_multiseed"
    ): (
        staging
        / "phase0"
        / "multiseed"
    ),

    Path(
        "/kaggle/working/phase0_evidential"
    ): (
        staging
        / "phase0"
        / "evidential_next_day"
    ),

    Path(
        "/kaggle/working/phase0_evidential_current"
    ): (
        staging
        / "phase0"
        / "evidential_current"
    ),

    Path(
        "/kaggle/working/phase0_router"
    ): (
        staging
        / "phase0"
        / "router"
    ),
}

for source, destination in backups.items():
    if not source.exists():
        raise FileNotFoundError(
            source
        )

    if destination.exists():
        shutil.rmtree(
            destination
        )

    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    shutil.copytree(
        source,
        destination,
    )

    print(
        f"Prepared: {source} -> {destination}"
    )

try:
    kagglehub.dataset_upload(
        HANDLE,
        str(staging),
        version_notes=(
            "Complete Phase 0: Neural CDE, "
            "multi-seed robustness, evidential probes, "
            "and routing ablation"
        ),
    )

    print(
        "✅ Complete Phase 0 safely backed up"
    )

except Exception as error:
    print(
        "Backup upload skipped:",
        error,
    )

Prepared: /kaggle/working/phase0_cde -> /kaggle/working/artifact_sync/phase0/neural_cde_rk4x4
Prepared: /kaggle/working/phase0_multiseed -> /kaggle/working/artifact_sync/phase0/multiseed
Prepared: /kaggle/working/phase0_evidential -> /kaggle/working/artifact_sync/phase0/evidential_next_day
Prepared: /kaggle/working/phase0_evidential_current -> /kaggle/working/artifact_sync/phase0/evidential_current
Prepared: /kaggle/working/phase0_router -> /kaggle/working/artifact_sync/phase0/router
Uploading Dataset https://kaggle.com/datasets/amirentezari/diffusion-portfolio ...
Starting upload for file /kaggle/working/artifact_sync/.artifact_store.json


Uploading: 100%|██████████| 129/129 [00:00<00:00, 177B/s]

Upload successful: /kaggle/working/artifact_sync/.artifact_store.json (129B)
Starting upload for file /kaggle/working/artifact_sync/phase0/neural_cde_rk4x4/best.pt



Uploading: 100%|██████████| 16.0M/16.0M [00:00<00:00, 17.6MB/s]

Upload successful: /kaggle/working/artifact_sync/phase0/neural_cde_rk4x4/best.pt (15MB)
Starting upload for file /kaggle/working/artifact_sync/phase0/neural_cde_rk4x4/summary.json


...


Uploading: 100%|██████████| 15.5M/15.5M [00:00<00:00, 17.8MB/s]

Upload successful: /kaggle/working/artifact_sync/baselines/vanilla_vpred/validation_scenarios.npz (15MB)
Starting upload for file /kaggle/working/artifact_sync/baselines/vanilla_vpred/validation_probabilistic_metrics.json



Uploading: 100%|██████████| 1.47k/1.47k [00:00<00:00, 2.04kB/s]

Upload successful: /kaggle/working/artifact_sync/baselines/vanilla_vpred/validation_probabilistic_metrics.json (1KB)


Backup upload skipped: 403 Client Error: Forbidden for url: https://api.kaggle.com/v1/datasets.DatasetApiService/CreateDatasetVersion
